# FireSight Data Pipeline

## 1. Imports and paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("raw")
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)
print("RAW:   ", RAW.resolve())
print("OUTPUT:", OUTPUT.resolve())

RAW:    C:\Users\Yehor Mykhailov\OneDrive - University of Glasgow\Second Year\Computing Science\Hackathons\Code for Good\Actual Hackathon\Project Name\Team-7\data-pipeline\raw
OUTPUT: C:\Users\Yehor Mykhailov\OneDrive - University of Glasgow\Second Year\Computing Science\Hackathons\Code for Good\Actual Hackathon\Project Name\Team-7\data-pipeline\output


## 2. Load raw datasets

Four datasets loaded conditionally. Missing files print a skip message and set their variable to `None`.
Only SIMD is required — all others are optional and the pipeline degrades gracefully.

In [2]:
# --- SIMD 2020 (REQUIRED) ---
simd_raw = pd.read_csv(RAW / "simd2020.csv")
print("SIMD shape:  ", simd_raw.shape)
print("SIMD columns:", simd_raw.columns.tolist())
simd_raw.head(3)

SIMD shape:   (6976, 16)
SIMD columns: ['DataZone', 'IntZone', 'HB', 'HSCP', 'CA', 'SIMD2020V2Rank', 'SIMD2020V2CountryDecile', 'SIMD2020V2CountryQuintile', 'SIMD2020V2HBDecile', 'SIMD2020V2HBQuintile', 'SIMD2020V2HSCPDecile', 'SIMD2020V2HSCPQuintile', 'SIMD2020V2CADecile', 'SIMD2020V2CAQuintile', 'SIMD2020V2Most15pc', 'SIMD2020V2Least15pc']


,DataZone,IntZone,HB,HSCP,CA,SIMD2020V2Rank,SIMD2020V2CountryDecile,SIMD2020V2CountryQuintile,SIMD2020V2HBDecile,SIMD2020V2HBQuintile,SIMD2020V2HSCPDecile,SIMD2020V2HSCPQuintile,SIMD2020V2CADecile,SIMD2020V2CAQuintile,SIMD2020V2Most15pc,SIMD2020V2Least15pc
0,S01006506,S02001236,S08000020,S37000001,S12000033,4691,7,4,5,3,6,3,6,3,0,0
1,S01006507,S02001236,S08000020,S37000001,S12000033,4862,7,4,6,3,6,3,6,3,0,0
2,S01006508,S02001236,S08000020,S37000001,S12000033,5686,9,5,8,4,7,4,7,4,0,0


In [3]:
# --- Census 2022 (optional) ---
# Expected columns: DataZone, total_population, age65plus,
#                   single_occ_households, total_households
# Adjust CENSUS_* constants in Section 3 to match actual column names.
CENSUS_PATH = RAW / "census2022glasgow.csv"
if CENSUS_PATH.exists():
    census_raw = pd.read_csv(CENSUS_PATH)
    print("Census shape:  ", census_raw.shape)
    print("Census columns:", census_raw.columns.tolist())
    census_raw.head(3)
else:
    print("census2022glasgow.csv not found — skipping Census")
    census_raw = None

census2022glasgow.csv not found — skipping Census


In [4]:
# --- SFRS Fire Incidents (optional) ---
# Expected columns: DataZone (or Postcode), IncidentDate, IncidentType
# Adjust INC_* constants in Section 3 to match actual column names.
INCIDENTS_PATH = RAW / "fireincidents20192025.csv"
if INCIDENTS_PATH.exists():
    incidents_raw = pd.read_csv(INCIDENTS_PATH)
    print("Incidents shape:  ", incidents_raw.shape)
    print("Incidents columns:", incidents_raw.columns.tolist())
    incidents_raw.head(3)
else:
    print("fireincidents20192025.csv not found — skipping Incidents")
    incidents_raw = None

fireincidents20192025.csv not found — skipping Incidents


In [5]:
# --- EPC Scotland (optional / stretch) ---
# Expected columns: DataZone, mean_construction_year, mean_epc_rating_numeric
#   mean_epc_rating_numeric: A=7, B=6, C=5, D=4, E=3, F=2, G=1
EPC_PATH = RAW / "epc_glasgow.csv"
if EPC_PATH.exists():
    epc_raw = pd.read_csv(EPC_PATH)
    print("EPC shape:  ", epc_raw.shape)
    print("EPC columns:", epc_raw.columns.tolist())
    epc_raw.head(3)
else:
    print("epc_glasgow.csv not found — skipping EPC")
    epc_raw = None

epc_glasgow.csv not found — skipping EPC


## 3. Clean and filter to Glasgow

Filter each loaded dataset to Glasgow City data zones (`CA == S12000049`).
Rename columns to canonical names used throughout the rest of the notebook.

In [6]:
GLASGOW_CA = "S12000049"

# Filter SIMD to Glasgow
before = len(simd_raw)
simd_gla = simd_raw[simd_raw["CA"] == GLASGOW_CA].copy()
after = len(simd_gla)
print(f"SIMD: {before} → {after} rows (Glasgow filter)")
assert after > 0, f"No rows matched CA == '{GLASGOW_CA}'. Check CA column values."

# Canonical column names
simd_clean = simd_gla[["DataZone", "IntZone", "SIMD2020V2CountryDecile"]].copy()
simd_clean.columns = ["zoneid", "zonename", "simddecile"]

# Build the canonical Glasgow zone list used for filtering other datasets
GLASGOW_ZONES = set(simd_clean["zoneid"])
print(f"Glasgow DataZones: {len(GLASGOW_ZONES)}")
print(f"Decile range: {simd_clean['simddecile'].min()} – {simd_clean['simddecile'].max()}")
simd_clean.head(3)

SIMD: 6976 → 746 rows (Glasgow filter)
Glasgow DataZones: 746
Decile range: 1 – 10


,zoneid,zonename,simddecile
3252,S01009758,S02001842,8
3253,S01009759,S02001842,1
3254,S01009760,S02001842,2


In [7]:
# --- Adjust these constants to match the actual Census CSV column names ---
CENSUS_ZONE_COL     = "DataZone"
CENSUS_POP_COL      = "total_population"
CENSUS_AGE65_COL    = "age65plus"
CENSUS_SINGLE_COL   = "single_occ_households"
CENSUS_TOTAL_HH_COL = "total_households"

if census_raw is not None:
    census_gla = census_raw[census_raw[CENSUS_ZONE_COL].isin(GLASGOW_ZONES)].copy()
    print(f"Census: {len(census_raw)} → {len(census_gla)} rows")
    assert len(census_gla) > 0, "Census filter returned 0 rows — check CENSUS_ZONE_COL"

    census_clean = census_gla[[CENSUS_ZONE_COL, CENSUS_POP_COL,
                                CENSUS_AGE65_COL, CENSUS_SINGLE_COL,
                                CENSUS_TOTAL_HH_COL]].copy()
    census_clean.columns = ["zoneid", "population", "age65plus",
                             "single_occ_households", "total_households"]

    # Derive percentage features
    census_clean["pctover65"] = (
        census_clean["age65plus"] / census_clean["population"] * 100
    ).round(2)
    census_clean["pctsingleocc"] = (
        census_clean["single_occ_households"] / census_clean["total_households"] * 100
    ).round(2)

    print(f"pctover65   — mean: {census_clean['pctover65'].mean():.1f}%  "
          f"nulls: {census_clean['pctover65'].isna().sum()}")
    print(f"pctsingleocc — mean: {census_clean['pctsingleocc'].mean():.1f}%  "
          f"nulls: {census_clean['pctsingleocc'].isna().sum()}")
    census_clean[["zoneid", "population", "pctover65", "pctsingleocc"]].head(3)
else:
    census_clean = None
    print("Census not loaded — population, pctover65, pctsingleocc will be NaN")

Census not loaded — population, pctover65, pctsingleocc will be NaN


In [8]:
# --- Adjust these constants to match the actual Incidents CSV column names ---
INC_ZONE_COL = "DataZone"   # or "Postcode" if no DataZone column
INC_DATE_COL = "IncidentDate"

if incidents_raw is not None:
    inc_gla = incidents_raw[incidents_raw[INC_ZONE_COL].isin(GLASGOW_ZONES)].copy()
    print(f"Incidents: {len(incidents_raw)} → {len(inc_gla)} rows (Glasgow filter)")
    dropout_pct = (1 - len(inc_gla) / len(incidents_raw)) * 100
    print(f"Dropout from filter: {dropout_pct:.1f}%")
    if dropout_pct > 30:
        print("WARNING: >30% dropout — check INC_ZONE_COL or whether postcode lookup is needed")

    inc_clean = inc_gla[[INC_ZONE_COL, INC_DATE_COL]].copy()
    inc_clean.columns = ["zoneid", "date"]
    inc_clean["date"] = pd.to_datetime(inc_clean["date"])
    print(f"Date range: {inc_clean['date'].min().date()} → {inc_clean['date'].max().date()}")
    inc_clean.head(3)
else:
    inc_clean = None
    print("Incidents not loaded — incident density and forecasts will be NaN")

Incidents not loaded — incident density and forecasts will be NaN


In [9]:
# --- Adjust these constants to match the actual EPC CSV column names ---
EPC_ZONE_COL   = "DataZone"
EPC_RATING_COL = "mean_epc_rating_numeric"   # A=7 (best) … G=1 (worst)

if epc_raw is not None:
    epc_gla = epc_raw[epc_raw[EPC_ZONE_COL].isin(GLASGOW_ZONES)].copy()
    print(f"EPC: {len(epc_raw)} → {len(epc_gla)} rows")
    coverage = len(epc_gla) / len(GLASGOW_ZONES) * 100
    print(f"DataZone coverage: {coverage:.1f}%")
    if coverage < 70:
        print("WARNING: EPC coverage <70% — rental stock likely underrepresented")

    epc_clean = epc_gla[[EPC_ZONE_COL, EPC_RATING_COL]].copy()
    epc_clean.columns = ["zoneid", "epc_rating"]
    # Invert so higher value = worse condition = higher fire risk
    epc_clean["epc_condition"] = 8 - epc_clean["epc_rating"]
    epc_clean[["zoneid", "epc_condition"]].head(3)
else:
    epc_clean = None
    print("EPC not loaded — housing condition component will be NaN")

EPC not loaded — housing condition component will be NaN


## 4. Feature engineering

Build one flat `features` table using SIMD zones as the spine. Left-join everything else.
Missing datasets leave their columns as NaN — the risk score handles this gracefully.

In [10]:
# SIMD spine
features = simd_clean[["zoneid", "zonename", "simddecile"]].copy()
features["simddecile_inv"] = 11 - features["simddecile"]  # higher = more deprived = higher risk

# Join Census
if census_clean is not None:
    features = features.merge(
        census_clean[["zoneid", "population", "pctover65", "pctsingleocc"]],
        on="zoneid", how="left"
    )
else:
    features["population"]   = np.nan
    features["pctover65"]    = np.nan
    features["pctsingleocc"] = np.nan

# Join incident density (total incidents per zone, computed from inc_clean)
if inc_clean is not None:
    total_inc = (
        inc_clean.groupby("zoneid").size().reset_index(name="total_incidents")
    )
    features = features.merge(total_inc, on="zoneid", how="left")
    features["total_incidents"] = features["total_incidents"].fillna(0)
    # incidents per 1,000 residents (requires population)
    mask = features["population"].notna() & (features["population"] > 0)
    features["incidents_per_1000"] = np.nan
    features.loc[mask, "incidents_per_1000"] = (
        features.loc[mask, "total_incidents"] / features.loc[mask, "population"] * 1000
    ).round(2)
else:
    features["total_incidents"]   = np.nan
    features["incidents_per_1000"] = np.nan

# Join EPC condition
if epc_clean is not None:
    features = features.merge(
        epc_clean[["zoneid", "epc_condition"]], on="zoneid", how="left"
    )
else:
    features["epc_condition"] = np.nan

# Summary
print("Features shape:", features.shape)
print("\nNull count per column:")
print(features.isnull().sum())
features.head(3)

Features shape: (746, 10)

Null count per column:
zoneid                  0
zonename                0
simddecile              0
simddecile_inv          0
population            746
pctover65             746
pctsingleocc          746
total_incidents       746
incidents_per_1000    746
epc_condition         746
dtype: int64


,zoneid,zonename,simddecile,simddecile_inv,population,pctover65,pctsingleocc,total_incidents,incidents_per_1000,epc_condition
3252,S01009758,S02001842,8,3,NaN,NaN,NaN,NaN,NaN,NaN
3253,S01009759,S02001842,1,10,NaN,NaN,NaN,NaN,NaN,NaN
3254,S01009760,S02001842,2,9,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Risk score computation

5-component weighted score. Each component is percentile-clipped (2nd–98th) then min-max normalised.
Score = weighted mean of available components × 100, rounded to integer.
Components with no data (NaN) are excluded from the weighted mean automatically.

In [11]:
RISK_WEIGHTS = {
    "simddecile_inv":     0.25,   # deprivation (higher decile_inv = more deprived)
    "pctover65":          0.20,   # elderly share
    "pctsingleocc":       0.15,   # single-occupancy rate
    "incidents_per_1000": 0.25,   # historical incident density per capita
    "epc_condition":      0.15,   # housing condition (higher = worse)
}

def normalise_component(series: pd.Series) -> pd.Series:
    """Clip to 2nd–98th percentile, then min-max normalise to [0, 1]."""
    valid = series.dropna()
    if len(valid) == 0:
        return series  # all NaN — leave as-is
    lo = valid.quantile(0.02)
    hi = valid.quantile(0.98)
    if hi == lo:  # zero variance — return 0.5 for all non-NaN
        return series.where(series.isna(), 0.5)
    clipped = series.clip(lo, hi)
    return (clipped - lo) / (hi - lo)

# Normalise each component into a _norm column
for col in RISK_WEIGHTS:
    if col in features.columns:
        features[f"{col}_norm"] = normalise_component(features[col])
    else:
        features[f"{col}_norm"] = np.nan

print("Normalised column null counts:")
norm_cols = [f"{c}_norm" for c in RISK_WEIGHTS]
print(features[norm_cols].isnull().sum())

Normalised column null counts:
simddecile_inv_norm          0
pctover65_norm             746
pctsingleocc_norm          746
incidents_per_1000_norm    746
epc_condition_norm         746
dtype: int64


In [12]:
def compute_riskscore(row):
    """Compute weighted risk score from available normalised components."""
    total_w = 0.0
    total_s = 0.0
    n = 0
    for col, w in RISK_WEIGHTS.items():
        norm_col = f"{col}_norm"
        val = row.get(norm_col, np.nan)
        if pd.notna(val):
            total_w += w
            total_s += w * val
            n += 1
    if total_w == 0 or n == 0:
        return pd.Series({"riskscore": np.nan, "n_score_components": 0})
    return pd.Series({
        "riskscore": int(round(total_s / total_w * 100)),
        "n_score_components": n
    })

score_result = features.apply(compute_riskscore, axis=1)
features["riskscore"]          = score_result["riskscore"]
features["n_score_components"] = score_result["n_score_components"].astype(int)

n_scored = features["riskscore"].notna().sum()
print(f"Zones with a risk score: {n_scored} / {len(features)}")
print(f"Components available per zone (mode): "
      f"{features['n_score_components'].mode().iloc[0]} of 5")
if n_scored > 0:
    print(f"Score range: {features['riskscore'].min():.0f} – {features['riskscore'].max():.0f}")
    print(f"Score mean:  {features['riskscore'].mean():.1f}")
    print("\nTop 10 highest-risk zones:")
    display(features[["zoneid", "zonename", "riskscore", "n_score_components"]]
            .sort_values("riskscore", ascending=False).head(10))
else:
    print("No risk scores computed yet — add Census and/or Incident data to unlock scoring")

Zones with a risk score: 746 / 746


Components available per zone (mode): 1 of 5
Score range: 0 – 100
Score mean:  68.2

Top 10 highest-risk zones:


,zoneid,zonename,riskscore,n_score_components
3269,S01009775,S02001845,100,1
3997,S01010503,S02001977,100,1
3270,S01009776,S02001845,100,1
3253,S01009759,S02001842,100,1
3601,S01010107,S02001904,100,1
3602,S01010108,S02001904,100,1
3604,S01010110,S02001905,100,1
3605,S01010111,S02001905,100,1
3606,S01010112,S02001905,100,1
3607,S01010113,S02001906,100,1


## 6. Drivers extraction

For each zone, rank the normalised components by weighted contribution and take the top 3.
Maps each component to a plain-English string for the frontend `drivers` field.

In [13]:
DRIVER_LABELS = {
    "simddecile_inv":     "High area deprivation",
    "pctover65":          "High proportion of elderly residents",
    "pctsingleocc":       "High single-occupancy rate",
    "incidents_per_1000": "Elevated historical incident rate",
    "epc_condition":      "Poor housing condition",
}

def extract_drivers(row):
    contributions = {}
    for col, w in RISK_WEIGHTS.items():
        val = row.get(f"{col}_norm", np.nan)
        if pd.notna(val):
            contributions[col] = val * w
    if not contributions:
        return None
    top3 = sorted(contributions, key=contributions.get, reverse=True)[:3]
    return [DRIVER_LABELS[c] for c in top3]

features["drivers"] = features.apply(extract_drivers, axis=1)

n_with_drivers = features["drivers"].notna().sum()
print(f"Zones with drivers: {n_with_drivers} / {len(features)}")
if n_with_drivers > 0:
    sample = features[features["drivers"].notna()][["zoneid", "drivers"]].head(5)
    display(sample)

Zones with drivers: 746 / 746


,zoneid,drivers
3252,S01009758,[High area deprivation]
3253,S01009759,[High area deprivation]
3254,S01009760,[High area deprivation]
3255,S01009761,[High area deprivation]
3256,S01009762,[High area deprivation]


## 7. Incident time-series

Aggregates incident records into 6-month period bins per zone.
Tags winter periods (Oct–Mar). Computes Glasgow-wide winter uplift factor.
Pivots to a zone × period matrix for the forecast model.

In [14]:
if inc_clean is not None:
    inc_ts = inc_clean.copy()

    # 6-month period label: YYYY-H1 (Jan–Jun) or YYYY-H2 (Jul–Dec)
    inc_ts["period"] = (
        inc_ts["date"].dt.year.astype(str)
        + "-H"
        + ((inc_ts["date"].dt.month > 6).astype(int) + 1).astype(str)
    )

    # Winter flag: Oct–Mar spans H2 of year N and H1 of year N+1.
    # We tag by month: Oct(10), Nov(11), Dec(12), Jan(1), Feb(2), Mar(3)
    inc_ts["is_winter"] = inc_ts["date"].dt.month.isin([10, 11, 12, 1, 2, 3])

    # Aggregate: incidents per zone per period, carry winter flag
    inc_agg = (
        inc_ts.groupby(["zoneid", "period"])
        .agg(incident_count=("date", "count"),
             winter_count=("is_winter", "sum"))
        .reset_index()
    )
    inc_agg["is_winter"] = inc_agg["winter_count"] > (inc_agg["incident_count"] / 2)

    # Glasgow-wide winter uplift factor
    w_mean  = inc_agg.loc[inc_agg["is_winter"],  "incident_count"].mean()
    s_mean  = inc_agg.loc[~inc_agg["is_winter"], "incident_count"].mean()
    WINTER_FACTOR = round(w_mean / s_mean, 3) if s_mean > 0 else 1.0
    print(f"Glasgow-wide winter factor: {WINTER_FACTOR:.3f}")
    print(f"  (>1.0 means more incidents in Oct–Mar than Jul–Sep)")

    # Pivot: zone × period, fill missing periods with 0
    pivot = (
        inc_agg.pivot_table(
            index="zoneid", columns="period",
            values="incident_count", fill_value=0
        )
    )
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)
    periods = pivot.columns.tolist()

    pct_sufficient = (pivot.shape[1] >= 4).sum() / len(pivot) * 100 if len(pivot) > 0 else 0
    print(f"\nPivot shape: {pivot.shape}  (zones × periods)")
    print(f"Periods: {periods}")
    print(f"Zones with ≥4 periods: {(pivot.notna().all(axis=1) & (pivot.shape[1] >= 4)).sum()}")

    # Determine if next forecast period falls in winter
    from datetime import date
    today_month = date.today().month
    NEXT_IS_WINTER = today_month in [10, 11, 12, 1, 2, 3, 4]  # within or approaching winter
    print(f"Next period is_winter: {NEXT_IS_WINTER}")
else:
    inc_agg     = None
    pivot       = None
    WINTER_FACTOR  = 1.0
    NEXT_IS_WINTER = False
    print("No incident data — time-series section skipped")

No incident data — time-series section skipped


## 8. Forecast model

Recency-weighted linear regression with:
- Exponential weights (recent periods count more)
- Additive winter seasonality adjustment
- Sparse-zone pooling (zones with <4 periods use Glasgow-wide median slope)

Produces 6 / 12 / 18 month forecasts with 95% residual confidence intervals.

In [15]:
def forecast_zone(
    counts,
    n_periods_ahead=(1, 2, 3),
    pooled_slope=None,
    winter_factor=1.0,
    next_is_winter=False,
    min_periods=4,
):
    """
    Recency-weighted linear forecast for a single zone.

    Parameters
    ----------
    counts         : 1-D array of 6-month incident counts (oldest → newest)
    n_periods_ahead: forecast horizons (1=6mo, 2=12mo, 3=18mo)
    pooled_slope   : Glasgow median slope — used when zone has too few periods
    winter_factor  : Glasgow-wide winter uplift ratio
    next_is_winter : whether the next forecast period is winter
    min_periods    : minimum periods required for zone's own trend fit

    Returns
    -------
    dict with forecast/lower/upper per horizon, trendpct, data_quality
    """
    counts = np.asarray(counts, dtype=float)
    n = len(counts)

    # --- Sparse zone fallback ---
    if n < min_periods:
        if pooled_slope is not None and n > 0:
            mean_level = float(np.mean(counts))
            results = {}
            for h in n_periods_ahead:
                raw = mean_level + pooled_slope * h
                pt  = max(0.0, raw)
                if next_is_winter:
                    pt *= winter_factor
                # Wider intervals for pooled estimate (±40%)
                lo = max(0.0, pt * 0.6)
                hi = pt * 1.4
                results[h] = {"forecast": round(pt), "lower": round(lo), "upper": round(hi)}
            latest = float(counts[-1])
            f6 = results[1]["forecast"]
            trendpct = round((f6 - latest) / latest * 100, 1) if latest > 0 else 0.0
            results["trendpct"]    = trendpct
            results["data_quality"] = "pooled"
        else:
            last = float(counts[-1]) if n > 0 else 0.0
            results = {
                h: {"forecast": round(last), "lower": round(last), "upper": round(last)}
                for h in n_periods_ahead
            }
            results["trendpct"]    = 0.0
            results["data_quality"] = "insufficient"
        return results

    # --- Recency-weighted linear fit ---
    x = np.arange(n, dtype=float)
    weights = np.exp(np.linspace(0.0, 1.0, n))   # more weight on recent periods
    slope, intercept = np.polyfit(x, counts, 1, w=weights)

    fitted       = slope * x + intercept
    residuals    = counts - fitted
    residual_std = float(np.std(residuals, ddof=1)) if n > 2 else float(np.abs(residuals).mean())

    results = {}
    for h in n_periods_ahead:
        raw = slope * (n - 1 + h) + intercept
        pt  = max(0.0, raw)
        if next_is_winter:
            pt *= winter_factor
        lo = max(0.0, pt - 1.96 * residual_std)
        hi = max(0.0, pt + 1.96 * residual_std)
        results[h] = {"forecast": round(pt), "lower": round(lo), "upper": round(hi)}

    latest   = float(counts[-1])
    f6       = results[1]["forecast"]
    trendpct = round((f6 - latest) / latest * 100, 1) if latest > 0 else 0.0

    results["trendpct"]    = trendpct
    results["residual_std"] = round(residual_std, 2)
    results["data_quality"] = "sufficient"
    return results


# === Smoke tests ===
print("Smoke test 1 — upward trend:")
r = forecast_zone(np.array([4, 5, 6, 7, 8, 9], dtype=float))
print(f"  6mo: {r[1]}  12mo: {r[2]}  18mo: {r[3]}  trend: {r['trendpct']}%  quality: {r['data_quality']}")

print("Smoke test 2 — flat series:")
r = forecast_zone(np.array([5, 5, 5, 5, 5, 5], dtype=float))
print(f"  6mo: {r[1]}  trend: {r['trendpct']}%")

print("Smoke test 3 — declining series (clamp to 0):")
r = forecast_zone(np.array([10, 7, 4, 2, 1, 0], dtype=float))
print(f"  6mo forecast: {r[1]['forecast']}  (should be 0)")

print("Smoke test 4 — sparse zone (2 periods, pooled fallback):")
r = forecast_zone(np.array([3, 4], dtype=float), pooled_slope=0.3)
print(f"  6mo: {r[1]}  quality: {r['data_quality']}")

Smoke test 1 — upward trend:
  6mo: {'forecast': 10, 'lower': 10, 'upper': 10}  12mo: {'forecast': 11, 'lower': 11, 'upper': 11}  18mo: {'forecast': 12, 'lower': 12, 'upper': 12}  trend: 11.1%  quality: sufficient
Smoke test 2 — flat series:
  6mo: {'forecast': 5, 'lower': 5, 'upper': 5}  trend: 0.0%
Smoke test 3 — declining series (clamp to 0):
  6mo forecast: 0  (should be 0)
Smoke test 4 — sparse zone (2 periods, pooled fallback):
  6mo: {'forecast': 4, 'lower': 2, 'upper': 5}  quality: pooled


In [16]:
# Compute Glasgow-wide pooled slope (median across zones with sufficient data)
if pivot is not None:
    zone_slopes = []
    for _, row in pivot.iterrows():
        c = row.values.astype(float)
        if len(c) >= 4:
            x = np.arange(len(c), dtype=float)
            w = np.exp(np.linspace(0, 1, len(c)))
            s, _ = np.polyfit(x, c, 1, w=w)
            zone_slopes.append(s)
    POOLED_SLOPE = float(np.median(zone_slopes)) if zone_slopes else 0.0
    print(f"Pooled slope (Glasgow median): {POOLED_SLOPE:.4f} incidents / period")
    print(f"  Based on {len(zone_slopes)} zones with ≥4 periods")
else:
    POOLED_SLOPE = 0.0
    print("No pivot data — pooled slope = 0 (no incidents loaded)")

No pivot data — pooled slope = 0 (no incidents loaded)


In [17]:
# Apply forecasts to all zones
if pivot is not None:
    forecast_rows = []
    for zoneid, row in pivot.iterrows():
        counts = row.values.astype(float)

        r = forecast_zone(
            counts,
            n_periods_ahead=[1, 2, 3],
            pooled_slope=POOLED_SLOPE,
            winter_factor=WINTER_FACTOR,
            next_is_winter=NEXT_IS_WINTER,
        )

        # Seasonality: % of all zone incidents that fall in winter months
        zone_rows = inc_agg[inc_agg["zoneid"] == zoneid]
        total_inc  = zone_rows["incident_count"].sum()
        winter_inc = zone_rows.loc[zone_rows["is_winter"], "incident_count"].sum()
        seasonality = round(winter_inc / total_inc * 100, 1) if total_inc > 0 else 0.0

        forecast_rows.append({
            "zoneid":               zoneid,
            "forecast6mo":          r[1]["forecast"],
            "forecastlower":        r[1]["lower"],
            "forecastupper":        r[1]["upper"],
            "forecast12mo":         r[2]["forecast"],
            "forecast18mo":         r[3]["forecast"],
            "trendpct":             r["trendpct"],
            "seasonalitywinterpct": seasonality,
            "data_quality":         r["data_quality"],
        })

    forecasts_df = pd.DataFrame(forecast_rows)
    print(f"Forecasts computed for {len(forecasts_df)} zones")
    print("Data quality breakdown:")
    print(forecasts_df["data_quality"].value_counts())
    forecasts_df.head(3)
else:
    forecasts_df = None
    print("No incident data — forecast columns will be NaN")

No incident data — forecast columns will be NaN


## 9. Assemble final output

Join features, risk score, drivers, and forecasts into one zone-level table.
Enforce the frontend schema column order. Run final quality checks.

In [18]:
# Start from features (includes riskscore, drivers, all feature columns)
zones = features[[
    "zoneid", "zonename", "simddecile",
    "riskscore", "drivers",
    "population", "pctover65", "pctsingleocc",
    "n_score_components"
]].copy()

# Join forecasts
if forecasts_df is not None:
    zones = zones.merge(forecasts_df, on="zoneid", how="left")
else:
    zones["forecast6mo"]          = np.nan
    zones["forecastlower"]        = np.nan
    zones["forecastupper"]        = np.nan
    zones["forecast12mo"]         = np.nan
    zones["forecast18mo"]         = np.nan
    zones["trendpct"]             = np.nan
    zones["seasonalitywinterpct"] = np.nan
    zones["data_quality"]         = "insufficient"

# Risk rank (1 = highest risk in Glasgow)
zones["riskrank"] = zones["riskscore"].rank(ascending=False, method="min").astype("Int64")

# Enforce frontend schema column order (internal columns appended at end)
SCHEMA_COLS = [
    "zoneid", "zonename", "riskscore", "drivers",
    "population", "pctover65", "pctsingleocc", "simddecile",
    "forecast6mo", "forecastlower", "forecastupper",
    "forecast12mo", "forecast18mo",
    "trendpct", "seasonalitywinterpct",
    # internal
    "riskrank", "n_score_components", "data_quality"
]
zones = zones[SCHEMA_COLS]

print("Final zones table shape:", zones.shape)
zones.head(3)

Final zones table shape: (746, 18)


,zoneid,zonename,riskscore,drivers,population,pctover65,pctsingleocc,simddecile,forecast6mo,forecastlower,forecastupper,forecast12mo,forecast18mo,trendpct,seasonalitywinterpct,riskrank,n_score_components,data_quality
3252,S01009758,S02001842,22,[High area deprivation],NaN,NaN,NaN,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,621,1,insufficient
3253,S01009759,S02001842,100,[High area deprivation],NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,insufficient
3254,S01009760,S02001842,89,[High area deprivation],NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,232,1,insufficient


In [19]:
print("=== Final output quality report ===")
print(f"Total zones: {len(zones)}")
print()

# Null counts for key columns
key_cols = ["zoneid", "simddecile", "riskscore", "forecast6mo",
            "population", "pctover65", "pctsingleocc"]
null_summary = zones[key_cols].isnull().sum()
print("Null counts (key columns):")
print(null_summary)
print()

# Score summary
if zones["riskscore"].notna().any():
    print(f"Risk score: min={zones['riskscore'].min():.0f}  "
          f"max={zones['riskscore'].max():.0f}  "
          f"mean={zones['riskscore'].mean():.1f}")
else:
    print("Risk score: all NaN — add Census + Incident data to compute scores")

# Data quality breakdown
print()
print("Data quality breakdown:")
print(zones["data_quality"].value_counts())

=== Final output quality report ===
Total zones: 746

Null counts (key columns):
zoneid            0
simddecile        0
riskscore         0
forecast6mo     746
population      746
pctover65       746
pctsingleocc    746
dtype: int64

Risk score: min=0  max=100  mean=68.2

Data quality breakdown:
data_quality
insufficient    746
Name: count, dtype: int64


## 10. Export outputs

In [20]:
csv_path = OUTPUT / "glasgowzones.csv"
zones.to_csv(csv_path, index=False)
print(f"Saved {len(zones)} zones to {csv_path.resolve()}")

Saved 746 zones to C:\Users\Yehor Mykhailov\OneDrive - University of Glasgow\Second Year\Computing Science\Hackathons\Code for Good\Actual Hackathon\Project Name\Team-7\data-pipeline\output\glasgowzones.csv


In [21]:
# Export GeoJSON — only if geopandas is available and boundary file exists
BOUNDARY_PATH = RAW / "glasgowdatazones.geojson"
try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False

if HAS_GEOPANDAS and BOUNDARY_PATH.exists():
    boundaries = gpd.read_file(BOUNDARY_PATH)
    print("Boundary file shape:", boundaries.shape)
    print("Boundary columns:", boundaries.columns.tolist())

    # Adjust this column name to match the actual boundary file zone ID column
    BOUNDARY_ZONE_COL = "DataZone"

    # Drop the drivers list column — GeoJSON doesn't handle lists cleanly
    zones_for_geo = zones.copy()
    zones_for_geo["drivers"] = zones_for_geo["drivers"].apply(
        lambda x: " | ".join(x) if isinstance(x, list) else ""
    )

    gdf = boundaries[[BOUNDARY_ZONE_COL, "geometry"]].merge(
        zones_for_geo, left_on=BOUNDARY_ZONE_COL, right_on="zoneid", how="left"
    )
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=boundaries.crs)

    geo_path = OUTPUT / "glasgowrisk.geojson"
    gdf.to_file(geo_path, driver="GeoJSON")
    print(f"Saved GeoJSON to {geo_path.resolve()}")
    print(f"GeoJSON zones: {len(gdf)}")
else:
    if not HAS_GEOPANDAS:
        print("geopandas not installed — skipping GeoJSON export")
    else:
        print(f"Boundary file not found at {BOUNDARY_PATH} — skipping GeoJSON export")
    print("GeoJSON export will run automatically once glasgowdatazones.geojson is added to raw/")

geopandas not installed — skipping GeoJSON export
GeoJSON export will run automatically once glasgowdatazones.geojson is added to raw/
